# 🧬 Lentiviral Insert Generator

Turn a TCR's **V gene, J gene, and CDR3** into a finished, colour-coded **annotated lentiviral insert** — right here in your browser. No installation on your own computer.

**How to use this notebook:**
1. Run the cells in order from top to bottom (click each cell and press **Shift+Enter**, or use *Runtime → Run all*).
2. The first cell installs Stitchr (takes ~1 minute).
3. The last cell shows a form — type your TCR, click **Generate Insert**, and the annotated document appears below it.

*Note: Google Colab resets when you close it, so you'll need to run the setup cell again each new session.*

In [ ]:
#@title Step 1 — Install Stitchr (run this first, ~1 minute) { display-mode: "form" }
# Installs Stitchr and downloads the human TCR gene data.
# This runs once per Colab session.
print("Installing Stitchr...")
!pip install stitchr IMGTgeneDL --quiet
print("Downloading human gene data...")
!stitchrdl -s human
print("\n✓ Setup complete. Now run the next two cells.")

In [ ]:
#@title Step 2 — Load the pipeline (run this cell) { display-mode: "form" }
import sys, os, re, subprocess, random, datetime
# ════════════════════════════════════════════════════════════════════
# CODON TABLE + SEQUENCE UTILITIES
# ════════════════════════════════════════════════════════════════════

CODONS = {
    'ATA':'I','ATC':'I','ATT':'I','ATG':'M','ACA':'T','ACC':'T','ACG':'T','ACT':'T',
    'AAC':'N','AAT':'N','AAA':'K','AAG':'K','AGC':'S','AGT':'S','AGA':'R','AGG':'R',
    'CTA':'L','CTC':'L','CTG':'L','CTT':'L','CCA':'P','CCC':'P','CCG':'P','CCT':'P',
    'CAC':'H','CAT':'H','CAA':'Q','CAG':'Q','CGA':'R','CGC':'R','CGG':'R','CGT':'R',
    'GTA':'V','GTC':'V','GTG':'V','GTT':'V','GCA':'A','GCC':'A','GCG':'A','GCT':'A',
    'GAC':'D','GAT':'D','GAA':'E','GAG':'E','GGA':'G','GGC':'G','GGG':'G','GGT':'G',
    'TCA':'S','TCC':'S','TCG':'S','TCT':'S','TTC':'F','TTT':'F','TTA':'L','TTG':'L',
    'TAC':'Y','TAT':'Y','TAA':'_','TAG':'_','TGC':'C','TGT':'C','TGA':'_','TGG':'W',
}
AAS = {}
for _c, _a in CODONS.items():
    AAS.setdefault(_a, []).append(_c)

def translate(seq, frame=0):
    seq = seq.upper()[frame:]
    return ''.join(CODONS.get(seq[i:i+3], '?') for i in range(0, (len(seq)//3)*3, 3))

def find_all(seq, sub):
    i, locs = 0, []
    while True:
        p = seq.find(sub, i)
        if p == -1: break
        locs.append(p); i = p + 1
    return locs

def find_restriction_sites(sites, seq):
    return {s: find_all(seq, sites[s]) for s in sites}

def swap_codons(s, site_name, idx):
    taboo = RESTRICTION_SITES[site_name]
    first = 3 * (idx // 3)
    last  = 3 * ((idx + len(taboo)) // 3)
    ci = random.choice(list(range(first, last + 3, 3)))
    codon = s[ci:ci+3].upper()
    aa = CODONS.get(codon, '?')
    nc = codon
    for cand in sorted(AAS.get(aa, [])):
        if cand != codon:
            nc = cand; break
    return s[:ci] + nc + s[ci+3:]

# ════════════════════════════════════════════════════════════════════
# INSERT GENERATION  (embedded, identical to generate_lentiviral_insert.py)
# ════════════════════════════════════════════════════════════════════

RESTRICTION_SITES = {'MfeI':'CAATTG','AccI':'GTATAC','NotI':'GCGGCCGC',
                     'SnaBI':'TACGTA','SpeI':'ACTAGT'}
CR = [
    'tctcgagta',
    'catgggctccaggctgctctgttgggtgctgctttgtctcctgggagcaggcccagtaaaggctgga',
    'aagatc',
    'acgtgacaccacccaaagtctcactgtttgagcctagcaaggcagaaattgccaacaagcagaagg'
    'ccaccctggtgtgcctggcaagagggttctttccagatcacgtggagctgtcctggtgggtcaacg'
    'gcaaagaagtgcattctggggtctgcaccgacccccaggcttacaaggagagtaattactcatattg'
    'tctgtcaagccggctgagagtgtccgccacattctggcacaaccctaggaatcatttccgctgccag'
    'gtccagtttcacggcctgagtgaggaagataaatggccagaggggtcacctaagccagtgacacaga'
    'acatcagcgcagaagcctggggacgagcagactgtggcattactagcgcctcctatcatcagggcgt'
    'gctgagcgccactatcctgtacgagattctgctgggaaaggccaccctgtatgctgtgctggtctcc'
    'ggcctggtgctgatggccatggtcaagaaaaagaactctgggagtggagccacaaatttctctctgc'
    'tgaaacaggctggagatgtggaggaaaaccccggccctatgaagagcctgcgcgtgctgctggtcat'
    'cctgtggctg',
    'tcgtgggtctggagccaa',
    'gacattcagaacccggaaccggct',
    'cagctgaaggacccccgatctcaggatagtactctgtgcctgttcaccgactttgatagtcagatca'
    'atgtgcctaaaaccatggaatccggaacttttattaccgacaagtgcgtgctggatatgaaagccat'
    'ggacagtaagtcaaacggcgccatcgcttggagcaatcagacatccttcacttgccaggatatcttc'
    'aaggagaccaacgcaacatacccatcctctgacgtgccctgtgatgccaccctgacagagaagtctt'
    'tcgaaacagacatgaacctgaattttcagaatctgagcgtgatgggcctgagaatcctgctgctgaa'
    'ggtcgctgggtttaatctgctgatgacactgcggctgtggtcctcatgaattcggaccgtgtccaat'
    'gtagc',
    'gtcgacaatcaacctctgga',
]

def trim_by_remainder(seq, factor=3, remainder=0):
    r = len(seq) % factor
    trimlen = r - remainder
    if trimlen < 0: trimlen += 3
    return seq if trimlen == 0 else seq[:-trimlen]

def generate_insert(achain, bchain):
    achain = trim_by_remainder(achain, 3, 0)
    bchain = trim_by_remainder(bchain, 3, 1)
    exp = {}
    ins = CR[0]
    exp['NotI'] = len(ins);  ins += RESTRICTION_SITES['NotI'] + CR[1] + bchain + CR[2]
    exp['SnaBI'] = len(ins); ins += RESTRICTION_SITES['SnaBI'] + CR[3]
    exp['MfeI'] = len(ins);  ins += RESTRICTION_SITES['MfeI'] + CR[4] + achain + CR[5]
    exp['AccI'] = len(ins);  ins += RESTRICTION_SITES['AccI'] + CR[6]
    exp['SpeI'] = len(ins);  ins += RESTRICTION_SITES['SpeI'] + CR[7]
    ins = ins.replace('aaggctggaaaggcta', 'aaggctgga')
    ins_aa = translate(ins, 0)
    guard = 0
    while guard < 200:
        guard += 1
        ok = True
        locs = find_restriction_sites(RESTRICTION_SITES, ins)
        for s in exp:
            for i in locs.get(s, []):
                if i != exp[s]:
                    ok = False
                    ins = swap_codons(ins, s, i)
        if ok: break
    ins = 'a' + ins
    return ins, ins_aa

# ════════════════════════════════════════════════════════════════════
# STITCHR + TRIMMING
# ════════════════════════════════════════════════════════════════════

STITCHR_CMDS = ['stitchr', '/opt/homebrew/bin/stitchr', '/usr/local/bin/stitchr']

def _extract_nt_from_stitchr(output):
    """
    Parse Stitchr's FASTA output and return the full nucleotide sequence.
    Stitchr wraps the sequence across many 60-char lines, so we must collect
    ALL sequence lines after the '>nt' header until the next blank line or
    the '>aa' header — not just the first line.
    """
    lines = output.splitlines()
    seq_lines = []
    capturing = False
    for ln in lines:
        s = ln.strip()
        if s.startswith('>'):
            if capturing:
                break                    # reached the >aa header — stop
            if s.lower().startswith('>nt') or ('nt|' in s.lower()):
                capturing = True
                continue
            # some builds emit a single '>' header; start capturing after it
            capturing = True
            continue
        if capturing:
            if s == '':
                if seq_lines:
                    break                # blank line ends the nt block
                continue
            if re.fullmatch(r'[ACGTNacgtn]+', s):
                seq_lines.append(s)
            else:
                # a non-sequence line (e.g. a divider) — stop if we have data
                if seq_lines:
                    break
    return ''.join(seq_lines).upper() if seq_lines else None


def run_stitchr(v, j, cdr3, locus):
    """Run stitchr -nl (no leader). Returns V/CDR3/J/CONSTANT nt string."""
    last = ""
    for base in STITCHR_CMDS:
        cmd = [base, '-v', v, '-j', j, '-cdr3', cdr3, '-s', 'HUMAN', '-l', locus, '-nl']
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=90)
        except (FileNotFoundError, subprocess.TimeoutExpired) as e:
            last = str(e); continue
        seq = _extract_nt_from_stitchr(r.stdout)
        if seq and len(seq) > 30:
            return seq
        last = (r.stdout + r.stderr)[:250]
    raise RuntimeError(
        f"Stitchr failed for {locus} ({v} / {j} / {cdr3}).\n\n"
        f"Message: {last}\n\n"
        "Check that Stitchr is installed (pip install stitchr) and that "
        "'stitchrdl -s HUMAN' has been run. If the gene name has a * or /, "
        "the app already passes it safely — but confirm the gene exists in IMGT.")

ALPHA_MARKERS = ['DIQNPDP','DIQNPEP','IQNPDPAV','IQNPEPAV','DIQNP']
# Human TRBC (beta constant) is highly conserved right after the opening E.
# These anchors are long and constant-region-specific so they cannot match
# inside the V/CDR3/J region by accident. We search for the anchor, then step
# back to the E that opens the constant region (keeping the terminal G in nt).
BETA_C_ANCHORS = ['DLNKVFPPEVAVF','DLRNVTPPKVSLF','DLKNVFPPEVAVF',
                  'DLNKVFPPEVAV','DLRNVTPPKVSL','DLKNVFPPEVAV',
                  'FPPEVAVFEPSE','PKVSLFEPSKAE']
# Fallbacks (shorter) only used if the long anchors all miss:
BETA_C_FALLBACK = ['EDLNKV','EDLRNV','EDLKNV']

def _aa_to_nt_cut(prot_index):
    """Convert an amino-acid index to the nucleotide index that starts it."""
    return prot_index * 3

def trim_alpha(nt):
    prot = translate(nt)
    for m in ALPHA_MARKERS:
        i = prot.find(m)
        if i != -1:
            return nt[:i*3]              # remainder 0
    p = nt.upper().find('ATATCCAGAA')
    if p == -1: p = nt.upper().find('ATAT')
    return nt[:p] if p != -1 else nt

def trim_beta(nt):
    """
    Cut off the human TRBC constant region, keeping the terminal G (first nt of
    the E codon that opens the constant region). Uses long, constant-specific
    anchors searched near the 3' end so a coincidental 'EDL' in the V/CDR3 region
    cannot cause an early cut.
    """
    prot = translate(nt)
    # 1) Try the long anchors. Each anchor begins with 'DL...' — the E is the
    #    residue immediately before it. Use the LAST occurrence (rightmost).
    for anchor in BETA_C_ANCHORS:
        i = prot.rfind(anchor)
        if i != -1:
            e_index = i - 1            # the E sits just before 'DL...'
            if e_index >= 0 and prot[e_index] == 'E':
                return nt[:e_index*3 + 1]   # keep the G (first nt of E codon)
            # if the residue before isn't E, still cut at the anchor's D, +1 for G
            return nt[:i*3 + 1]
    # 2) Fallback: the short 'EDLxxx' markers, but taken as the LAST occurrence
    #    and only if they appear in the last ~40% of the protein (constant region
    #    is always near the 3' end) to avoid a spurious early hit.
    min_pos = int(len(prot) * 0.55)
    best = -1
    for m in BETA_C_FALLBACK:
        j = prot.rfind(m)
        if j >= min_pos and j > best:
            best = j
    if best != -1:
        return nt[:best*3 + 1]         # 'E' is first residue of marker; keep its G
    # 3) Last resort: return unchanged (caller will warn on remainder/G check)
    return nt

# ════════════════════════════════════════════════════════════════════
# ANNOTATION -> HTML (colour-coded, opens in browser, prints to PDF)
# ════════════════════════════════════════════════════════════════════

COLORS = {
    'notI':'#B8860B','beta':'#1B5E20','linker':'#37474F','mfei':'#6A1B9A',
    'alpha':'#BF360C','bstz17i':'#B71C1C','spei':'#00695C',
    'bracket':'#B71C1C','gray':'#9E9E9E',
}

def annotate_segments(insert_nt):
    s = insert_nt
    segs = []
    bracket_start = s.find('GCGGCCGC') - len(CR[0]) - 1
    segs.append((s[:bracket_start], COLORS['gray']))
    segs.append(('[', COLORS['bracket']))
    rest = s[bracket_start+1:]
    def take(n, col):
        nonlocal rest
        segs.append((rest[:n], col)); rest = rest[n:]
    take(len(CR[0]), COLORS['linker'])
    take(8, COLORS['notI'])
    take(len(CR[1]), COLORS['linker'])
    take(rest.find(CR[2]), COLORS['beta'])
    take(len(CR[2] + RESTRICTION_SITES['SnaBI'] + CR[3]), COLORS['linker'])
    take(6, COLORS['mfei'])
    take(len(CR[4]), COLORS['linker'])
    take(rest.find(CR[5]), COLORS['alpha'])
    take(len(CR[5]), COLORS['linker'])
    take(6, COLORS['bstz17i'])
    take(10, COLORS['linker'])          # first 10 nt of muTrac (inside bracket)
    segs.append((']', COLORS['bracket']))   # close synthesis region here (matches reference)
    take(len(CR[6]) - 10, COLORS['gray'])   # rest of muTrac (outside bracket)
    take(6, COLORS['gray'])                 # SpeI (outside)
    segs.append((rest, COLORS['gray']))     # CR[7] tail (outside)
    return segs

def seam_check(insert_aa, cdr3_b):
    i = insert_aa.find(cdr3_b)
    if i == -1:
        return "CDR3 not found", False
    seam = insert_aa[i+len(cdr3_b):i+len(cdr3_b)+16]
    # Correct seam has the glutamate (E) opening the constant region: ...E DLRNV / DLKNV
    ok = ('EDLR' in seam or 'EDLK' in seam or 'EDLN' in seam)
    return seam, ok

def build_html(name, desc, system,
               trav, traj, cdr3_a, trbv, trbj, cdr3_b,
               insert_nt, insert_aa):
    segs = annotate_segments(insert_nt)
    seam, ok = seam_check(insert_aa, cdr3_b)

    # coloured nucleotide HTML (monospace, wraps naturally)
    nt_html = ''
    for text, col in segs:
        if not text: continue
        safe = text.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
        weight = 'bold' if col in (COLORS['notI'],COLORS['bstz17i'],
                                   COLORS['spei'],COLORS['bracket'],COLORS['mfei']) else 'normal'
        nt_html += f"<span style='color:{col};font-weight:{weight}'>{safe}</span>"

    aa_main = insert_aa.split('_')[0]
    aa_tail = insert_aa[len(aa_main):]

    seam_color = '#1B5E20' if ok else '#B71C1C'
    seam_word  = 'PASSED' if ok else 'CHECK'

    legend = [
        (COLORS['bracket'], '[ ]', 'Genscript synthesis region'),
        (COLORS['notI'],   'NotI', "5' cloning site"),
        (COLORS['beta'],   'Beta (TRB)', 'Clone-specific beta chain'),
        (COLORS['linker'], 'Scaffold', 'Signal peptides, muTrbc, P2A, muTrac'),
        (COLORS['mfei'],   'MfeI', 'Internal site'),
        (COLORS['alpha'],  'Alpha (TRA)', 'Clone-specific alpha chain'),
        (COLORS['bstz17i'],'BstZ17I', "3' cloning site"),
        (COLORS['spei'],   'SpeI', 'Internal site'),
        (COLORS['gray'],   'Gray', 'Vector context (not synthesised)'),
    ]
    legend_html = ''
    for col, lab, d in legend:
        legend_html += (f"<div class='key'><span class='sw' style='background:{col}'></span>"
                        f"<b style='color:{col}'>{lab}</b> &nbsp;{d}</div>")

    today = datetime.date.today().isoformat()
    return f"""<!DOCTYPE html><html><head><meta charset='utf-8'>
<title>{name} — Annotated Insert</title>
<style>
  body {{ font-family: Arial, Helvetica, sans-serif; margin: 40px; color:#12293d; }}
  h1 {{ font-size: 20px; margin-bottom: 2px; }}
  .meta {{ font-size: 13px; color:#444; margin: 2px 0; }}
  .desc {{ font-style: italic; color:#666; font-size:13px; }}
  .seam {{ font-weight:bold; font-size:12.5px; margin:8px 0; color:{seam_color}; }}
  hr {{ border:none; border-top:1px solid #c0cdd8; margin:12px 0; }}
  .seq {{ font-family: 'Courier New', monospace; font-size: 12px;
          line-height: 1.55; word-break: break-all; }}
  .sectitle {{ font-weight:bold; color:#1B3A5C; margin:14px 0 4px; font-size:13px; }}
  .key {{ font-size:12px; margin:2px 0; }}
  .sw {{ display:inline-block; width:11px; height:11px; border-radius:2px;
         margin-right:5px; vertical-align:middle; }}
  .legendbox {{ columns:2; margin:10px 0; }}
  .tail {{ color:#bbb; }}
  @media print {{ body {{ margin: 18px; }} }}
</style></head><body>
<h1>Annotated Lentiviral Insert — {name}</h1>
<div class='desc'>{desc}</div>
<div class='meta'><b>TRA:</b> {trav} / {traj} / <span style="font-family:monospace">{cdr3_a}</span>
&nbsp;&nbsp; <b>TRB:</b> {trbv} / {trbj} / <span style="font-family:monospace">{cdr3_b}</span></div>
<div class='meta'>{system} &nbsp;·&nbsp; generated {today}</div>
<div class='seam'>Seam check {seam_word}: beta J/constant junction reads
…<span style="font-family:monospace">{cdr3_b}·{seam}</span>…</div>
<hr>
<div class='legendbox'>{legend_html}</div>
<hr>
<div class='sectitle'>Annotated nucleotide sequence</div>
<div class='seq'>{nt_html}</div>
<div class='sectitle'>Translated amino acid sequence</div>
<div class='seq'>{aa_main}<span class='tail'>{aa_tail}</span></div>
<hr>
<div style='font-size:11px;color:#888;font-style:italic'>
Generated by tcr_insert_gui.py · Insert length {len(insert_nt)} nt ·
To save as PDF: File → Print → Save as PDF</div>
</body></html>"""

# ════════════════════════════════════════════════════════════════════
# PIPELINE (used by the GUI button)
# ════════════════════════════════════════════════════════════════════


print('✓ Pipeline loaded.')

In [ ]:
#@title Step 3 — Generate your insert { display-mode: "form" }
# Fill in the fields below and click "Generate Insert".
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import os

# Input fields
w_name   = widgets.Text(value="MyClone", description="Sample:", style={'description_width':'90px'}, layout=widgets.Layout(width='420px'))
w_system = widgets.Text(value="DR4 / clone 461", description="System:", style={'description_width':'90px'}, layout=widgets.Layout(width='420px'))
w_desc   = widgets.Text(value="", description="Notes:", style={'description_width':'90px'}, layout=widgets.Layout(width='420px'))

w_trav = widgets.Text(value="TRAV41", description="TRAV:", style={'description_width':'90px'}, layout=widgets.Layout(width='300px'))
w_traj = widgets.Text(value="TRAJ49", description="TRAJ:", style={'description_width':'90px'}, layout=widgets.Layout(width='300px'))
w_cdr3a= widgets.Text(value="CAAAGNQFYF", description="TRA CDR3:", style={'description_width':'90px'}, layout=widgets.Layout(width='420px'))

w_trbv = widgets.Text(value="TRBV19", description="TRBV:", style={'description_width':'90px'}, layout=widgets.Layout(width='300px'))
w_trbj = widgets.Text(value="TRBJ2-3", description="TRBJ:", style={'description_width':'90px'}, layout=widgets.Layout(width='300px'))
w_cdr3b= widgets.Text(value="CASGRSHGTDTQYF", description="TRB CDR3:", style={'description_width':'90px'}, layout=widgets.Layout(width='420px'))

btn = widgets.Button(description="Generate Insert", button_style='primary',
                     layout=widgets.Layout(width='200px', height='38px'))
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        name   = w_name.value.strip() or "insert"
        system = w_system.value.strip()
        desc   = w_desc.value.strip()
        trav, traj, cdr3a = w_trav.value.strip(), w_traj.value.strip(), w_cdr3a.value.strip().upper()
        trbv, trbj, cdr3b = w_trbv.value.strip(), w_trbj.value.strip(), w_cdr3b.value.strip().upper()

        print(f"► {name}")
        print(f"  TRA: {trav} / {traj} / {cdr3a}")
        print(f"  TRB: {trbv} / {trbj} / {cdr3b}")
        print("  Running Stitchr and building insert...")
        try:
            a_full = run_stitchr(trav, traj, cdr3a, 'TRA')
            b_full = run_stitchr(trbv, trbj, cdr3b, 'TRB')
            a_trim = trim_alpha(a_full)
            b_trim = trim_beta(b_full)
            warns = []
            if len(a_trim) % 3 != 0: warns.append(f"alpha remainder {len(a_trim)%3} (expected 0)")
            if len(b_trim) % 3 != 1: warns.append(f"beta remainder {len(b_trim)%3} (expected 1)")
            if not b_trim.upper().endswith('G'): warns.append("beta does not end in G")
            ins_nt, ins_aa = generate_insert(a_trim, b_trim)
            if cdr3a not in ins_aa: warns.append(f"alpha CDR3 {cdr3a} not found")
            if cdr3b not in ins_aa: warns.append(f"beta CDR3 {cdr3b} not found")
            seam, ok = seam_check(ins_aa, cdr3b)
            print(f"  Beta seam: ...{seam}  [{'OK' if ok else 'CHECK THIS'}]")
            for wn in warns: print("  \u26a0 " + wn)
            print(f"  Insert length: {len(ins_nt)} nt")

            # Save files to the Colab session (downloadable from the Files panel)
            safe = "".join(c if c.isalnum() or c in "_-." else "_" for c in name)
            html = build_html(name, desc, system, trav, traj, cdr3a, trbv, trbj, cdr3b, ins_nt, ins_aa)
            with open(f"{safe}_annotated_insert.html","w") as f: f.write(html)
            with open(f"{safe}_output.txt","w") as f:
                f.write(f"Sample: {name}\nSystem: {system}\n")
                f.write(f"TRA: {trav} / {traj} / {cdr3a}\nTRB: {trbv} / {trbj} / {cdr3b}\n\n")
                f.write("Insert nucleotide sequence:\n"+ins_nt+"\n\nInsert amino acid sequence:\n"+ins_aa+"\n")
            print(f"  \u2713 Saved {safe}_annotated_insert.html and {safe}_output.txt (see the Files panel on the left)")
            print("\n" + "="*60 + "\n")
            display(HTML(html))
        except Exception as ex:
            print("  \u2717 ERROR:", ex)

btn.on_click(on_click)

display(widgets.HTML("<h3 style='color:#12293d'>TCR Lentiviral Insert Generator</h3>"
                     "<p style='color:#555'>Fill in the six values and click Generate Insert.</p>"))
display(widgets.HTML("<b style='color:#BF360C'>Alpha chain (TRA)</b>"))
display(w_trav, w_traj, w_cdr3a)
display(widgets.HTML("<b style='color:#1B5E20'>Beta chain (TRB)</b>"))
display(w_trbv, w_trbj, w_cdr3b)
display(widgets.HTML("<b>Labels</b>"))
display(w_name, w_system, w_desc)
display(btn, out)